# Binary Classification Neural Network from Scratch in NumPy

This notebook implements a single-neuron feedforward artificial neural network from scratch using Python and NumPy without deep learning frameworks. It derives and computes analytical gradients using the chain rule, trains via Stochastic Gradient Descent (SGD), and visualizes the learned decision boundary in 2D feature space.

In [ ]:
%matplotlib inline
import numpy as np
from matplotlib import pyplot as plt

## 1. Dataset Preparation

Each training sample consists of two morphological continuous features:
- Feature 1: Length
- Feature 2: Width
- Binary Target: 0 (Blue class) or 1 (Red class)

A mystery query sample  is designated for out-of-sample inference.

In [ ]:
# Training dataset: [length, width, class_label]
data = [
    [3,   1.5, 1],
    [2,   1,   0],
    [4,   1.5, 1],
    [3,   1,   0],
    [3.5, 0.5, 1],
    [2,   0.5, 0],
    [5.5, 1,   1],
    [1,   1,   0]
]

mystery_flower = [4.5, 1.0]

## 2. Exploratory Data Visualization

Visualizing the training distribution along with the unlabelled query point in 2D Euclidean feature space.

In [ ]:
def vis_data():
    plt.figure(figsize=(7, 5))
    plt.grid(True, linestyle="--", alpha=0.6)
    for point in data:
        color = "r" if point[2] == 1 else "b"
        plt.scatter(point[0], point[1], c=color, s=70, edgecolors="k")
    plt.scatter(mystery_flower[0], mystery_flower[1], c="gray", s=90, marker="*", label="Mystery Sample")
    plt.xlabel("Petal Length")
    plt.ylabel("Petal Width")
    plt.title("Input Data Distribution (Red = 1, Blue = 0, Star = Query)")
    plt.legend()
    plt.show()

vis_data()

## 3. Activation Function & Derivative

The logistic sigmoid non-linearity is defined as:
52031\sigma(z) = rac{1}{1 + e^{-z}}52031

Its first-order derivative expressed in terms of the activation itself:
52031\sigma'(z) = \sigma(z)(1 - \sigma(z))52031

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_p(x):
    s = sigmoid(x)
    return s * (1 - s)

X = np.linspace(-6, 6, 200)
plt.figure(figsize=(7, 4))
plt.plot(X, sigmoid(X), label=r"Sigmoid: $\sigma(x)$", color="blue")
plt.plot(X, sigmoid_p(X), label=r"Derivative: $\sigma'(x)$", color="red", linestyle="--")
plt.title("Sigmoid Activation Function and First Derivative")
plt.xlabel("z")
plt.ylabel("Output")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

## 4. Backpropagation & Training Loop (SGD)

The network calculates loss via squared error:
52031C = (\hat{y} - y)^252031

Gradients derived via the Chain Rule:
52031rac{\partial C}{\partial w_i} = rac{\partial C}{\partial \hat{y}} \cdot rac{\partial \hat{y}}{\partial z} \cdot rac{\partial z}{\partial w_i} = 2(\hat{y} - y) \cdot \sigma'(z) \cdot x_i52031
52031rac{\partial C}{\partial b} = rac{\partial C}{\partial \hat{y}} \cdot rac{\partial \hat{y}}{\partial z} \cdot rac{\partial z}{\partial b} = 2(\hat{y} - y) \cdot \sigma'(z) \cdot 152031

In [ ]:
def train(iterations=20000, learning_rate=0.1):
    np.random.seed(42)
    w1 = np.random.randn()
    w2 = np.random.randn()
    b = np.random.randn()
    costs = []

    for i in range(iterations):
        ri = np.random.randint(len(data))
        point = data[ri]
        z = point[0] * w1 + point[1] * w2 + b
        pred = sigmoid(z)
        target = point[2]

        if i % 100 == 0:
            total_cost = sum(np.square(sigmoid(p[0] * w1 + p[1] * w2 + b) - p[2]) for p in data)
            costs.append(total_cost)

        dcost_dpred = 2 * (pred - target)
        dpred_dz = sigmoid_p(z)
        dcost_dz = dcost_dpred * dpred_dz

        dcost_dw1 = dcost_dz * point[0]
        dcost_dw2 = dcost_dz * point[1]
        dcost_db = dcost_dz * 1.0

        w1 -= learning_rate * dcost_dw1
        w2 -= learning_rate * dcost_dw2
        b -= learning_rate * dcost_db

    return costs, w1, w2, b

costs, w1, w2, b = train()

plt.figure(figsize=(7, 4))
plt.plot(costs, color="teal")
plt.title("Loss Minimization Curve during Stochastic Gradient Descent")
plt.xlabel("Iterations (x100)")
plt.ylabel("Cumulative Squared Error")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()
print(f"Optimized Parameters: w1={w1:.4f}, w2={w2:.4f}, b={b:.4f}")

## 5. Mystery Flower Prediction

Evaluating the unlabelled sample  using the learned parameter weights.

In [ ]:
z = w1 * mystery_flower[0] + w2 * mystery_flower[1] + b
pred = sigmoid(z)
predicted_class = "Red (1)" if pred >= 0.5 else "Blue (0)"
print(f"Mystery Sample: {mystery_flower}")
print(f"Raw Sigmoid Probability: {pred:.4f}")
print(f"Predicted Classification: {predicted_class}")

## 6. Learned Decision Boundary in Feature Space

Mapping the network predictions across a 2D dense coordinate grid to reveal the linear decision boundary dividing class regions.

In [ ]:
plt.figure(figsize=(8, 6))
grid_x = np.linspace(0, 6, 40)
grid_y = np.linspace(0, 3, 40)
for x in grid_x:
    for y in grid_y:
        pred_val = sigmoid(w1 * x + w2 * y + b)
        c = "red" if pred_val >= 0.5 else "blue"
        plt.scatter(x, y, color=c, alpha=0.15, s=20)

for point in data:
    color = "red" if point[2] == 1 else "blue"
    plt.scatter(point[0], point[1], color=color, s=80, edgecolors="black", linewidth=1.5)

plt.scatter(mystery_flower[0], mystery_flower[1], color="yellow", edgecolors="black", s=120, marker="*", label="Mystery Flower")
plt.xlabel("Petal Length")
plt.ylabel("Petal Width")
plt.title("Learned 2D Decision Boundary & Classification Grid")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()